# Implementation of sleep scoring from Watson et al., Neuron (2016)
The paper uses three data sources to classify arousal stages into Awake/NREM/REM/Microarousal, and we want to attempt a further classification of Awake into Active and Quite Wakefulness.

Three data sources are:
- Broadband LFP
- Narrowband theta
- Intracranial EMG

All analyses were originally carried out on ephys recordings from neocortex, downsampled to 1250Hz.

**Broadband LFP** (Low-frequency spectral features)
1. Short-time FFT: 10s window, 1s step, evaluated at log-spaced frequencies between 1–100 Hz
2. z-score
3. PCA
4. Pick PC1. If negative loading on <20Hz frequencies, flip sign.
5. Pick trough in bimodal distribution as threshold between NREM and Wake/REM.

**Narrowband theta**
- Ratio of 5-10Hz/2-16Hz power. Hippocampal channels used where present.

**Intracranial EMG**
1. Bandpass (Butterworth, 4th order) filter in 300-600Hz range
2. Subselect pairs of random, good channels ≥ 2 shanks apart
3. Take mean of Pearson's R in 500ms windows --> **this is the EMG score**

### Imports

In [ ]:
import os
import yaml
import warnings

import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
import spikeinterface.full as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as sp

from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from scipy.stats import gaussian_kde
from diptest import diptest
from probeinterface import read_probeinterface
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

from sleep_sandbox.io import (
    find_amplifier_files, remote_available, get_or_build, load_preprocessed,
    load_bno055, concat_bno055, align_bno055_to_lfp,
)
from sleep_sandbox.preprocessing import build_band, imu_kinematics
from sleep_sandbox.analysis import (
    log_spectrogram, broadband_pc1, select_channel_by_dip, pc1_threshold,
    THETA_BANDS, band_power, theta_ratio, select_theta_channel, zscore, bin_max,
)


### Load configs and set up recording object

In [ ]:
load_dotenv()
repo_root = Path(find_dotenv()).parent

remote_data_dir = os.getenv("REMOTE_DATA_DIR")
derivatives_dir = os.getenv("DERIVATIVES_DIR")
local_parent = os.getenv("LOCAL_DERIV_PARENT")

# Recording selection: which probe + which AmplifierData _N segment(s) to concatenate.
probe_name = "ProbeB"
suffixes = None                              # int / range / list of AmplifierData _N segments

# Streams the pipeline needs; local mode loads these, remote mode builds them.
streams = ["lfp_nocmr", "lfp_cmr", "emg"]

# Local-first: use local derivatives if all present, even when the remote is mounted.
local_parent = Path(local_parent) if local_parent else None
use_local = local_parent is not None and all(any(local_parent.glob(f"{s}__*")) for s in streams)

if use_local:
    print(f"Local derivatives found; loading from {local_parent}")
elif remote_available(remote_data_dir):
    warnings.warn("No complete set of local derivatives; building from remote raw data.")
else:
    raise RuntimeError(
        "No local derivatives and REMOTE_DATA_DIR not available. "
        "Set LOCAL_DERIV_PARENT to a folder of prebuilt derivatives, or mount the remote raw data."
    )

In [ ]:
# Remote build path: discover raw segments, load config, concatenate. Skipped when use_local.
if not use_local:
    session_dir = Path(remote_data_dir).parent   # datetime dir holding probe config + NeuropixelsV2/

    try:
        amplifier_paths, probe_config_path = find_amplifier_files(session_dir, probe_name, suffixes)
    except FileNotFoundError:
        probe_config_path = Path(os.getenv("PROBE_CONFIG_PATH"))

    # Concatenation is only valid for a contiguous run of segments (no gaps).
    seg_ns = sorted(int(p.stem.split("_")[-1]) for p in amplifier_paths)
    assert seg_ns == list(range(seg_ns[0], seg_ns[-1] + 1)), f"non-contiguous segments: {seg_ns}"
    seg_source = f"{probe_name}_seg{seg_ns[0]}-{seg_ns[-1]}"

    # Load preprocessing config
    config_path = os.path.join(repo_root, "config/preprocessing.yml")
    with open(config_path) as f:
        config = yaml.safe_load(f)

    sample_rate = config['recording']['sample_rate']
    n_channels = config['recording']['n_channels']
    dtype = config['recording']['dtype']
    freq_min = config['lfp']['bandpass_filter']['freq_min']
    freq_max = config['lfp']['bandpass_filter']['freq_max']
    freq_min_highpass = config['preprocessing']['bandpass_filter']['freq_min']
    freq_max_highpass = config['preprocessing']['bandpass_filter']['freq_max']
    reference = config['lfp']['common_reference']['reference']
    operator = config['lfp']['common_reference']['operator']
    resample_rate = config['lfp']['resample_rate']

    # Concatenate the selected segments into one continuous recording (bridges internal boundaries).
    segments = [se.read_binary(p, sample_rate, dtype, n_channels) for p in amplifier_paths]
    recording = si.concatenate_recordings(segments)

### Add probe

In [ ]:
if not use_local:
    probe_group = read_probeinterface(probe_config_path)
    probe = probe_group.probes[0]

    active_mask = probe.device_channel_indices != -1
    active_probe = probe.get_slice(active_mask)

    # active_probe has 384 contacts, all mapped 1:1 to recording channels.
    # The full probe has 5120 contacts (384 active); copy_metadata in preprocessors silently
    # drops it when contact count != channel count, causing the "no Probe attached" warning.
    recording_with_probe = recording.set_probe(active_probe)

### Preprocess

We want to filter this in two ways for a first pass:
- Broadband LFP, roughly 0.5-300 Hz
- Intracranial EMG band, 300-600 Hz

In [ ]:
# Preprocessing chains (lazy - no data loaded yet). build_band returns each recording together
# with its canonical step list, so the chain and the params keying its derivative can't desync.
# margin_ms must be long enough for the 0.25 Hz high-pass corner (settles over seconds).
if not use_local:
    margin_ms = 12000

    # Shared input provenance: the exact ordered concatenated segments, pinned into every hash.
    concat_step = ("concatenate", {"inputs": [p.stem for p in amplifier_paths]})

    # LFP band ± CMR, and EMG band
    recording_lfp_no_cmr, lfp_nocmr_steps = build_band(
        recording_with_probe, freq_min, freq_max, resample_rate, margin_ms, source_steps=[concat_step])
    recording_lfp, lfp_cmr_steps = build_band(
        recording_with_probe, freq_min, freq_max, resample_rate,  margin_ms,
        cmr=True, reference=reference, operator=operator, source_steps=[concat_step])
    recording_emg, emg_steps = build_band(
        recording_with_probe, 300, 600, resample_rate, margin_ms, source_steps=[concat_step])

In [ ]:
if not use_local:
    preprocessed_dir = Path(derivatives_dir) / "preprocessed"

    # Each call loads from disk if the param-matched folder exists, else builds + saves it.
    # steps come straight from build_band above, so they always mirror the actual chain.
    recording_lfp_no_cmr = get_or_build(recording_lfp_no_cmr, seg_source, lfp_nocmr_steps, preprocessed_dir, "lfp_nocmr", n_jobs=4, chunk_duration="30s")
    recording_lfp = get_or_build(recording_lfp, seg_source, lfp_cmr_steps, preprocessed_dir, "lfp_cmr", n_jobs=20, chunk_duration="30s")
    recording_emg = get_or_build(recording_emg, seg_source, emg_steps, preprocessed_dir, "emg", n_jobs=6, chunk_duration="30s")

In [ ]:
# Local load path: read prebuilt derivatives by stream name (no raw / no mount needed).
if use_local:
    recording_lfp_no_cmr = load_preprocessed(local_parent, "lfp_nocmr")
    recording_lfp = load_preprocessed(local_parent, "lfp_cmr")
    recording_emg = load_preprocessed(local_parent, "emg")

### Dominant spectral feature in broadband LFP

In [ ]:
# Slow-wave analysis functions (log_spectrogram, broadband_pc1, select_channel_by_dip,
# pc1_threshold) are imported from sleep_sandbox.analysis -- see the imports cell.


In [ ]:
# Channel selection: scan every Nth channel, score PC1 bimodality with Hartigan's dip test.
channel_stride = 16     # scan every Nth channel
manual_channel = None   # set to an int channel index to override the dip-test choice

rec = recording_lfp
fs = rec.get_sampling_frequency()

best_channel, dip_stats, candidate_idx = select_channel_by_dip(rec, fs, channel_stride, manual_channel)
print(f"Selected channel {best_channel} (dip={dip_stats[best_channel]:.4f})")

In [ ]:
# Bimodality (Hartigan dip) across scanned channels
plt.figure(figsize=(10, 3))
plt.plot(candidate_idx, dip_stats[candidate_idx], marker=".")
plt.axvline(best_channel, color="r", ls="--", label=f"best = {best_channel}")
plt.xlabel("channel index")
plt.ylabel("dip statistic")
plt.legend()

In [ ]:
# Run the broadband slow-wave pipeline on the selected channel.
sw_channel = rec.channel_ids[best_channel]
sw_trace = rec.get_traces(channel_ids=[sw_channel]).squeeze()
sw_pc1, sw_pc1_loading, sw_spec, sw_freqs, sw_times = broadband_pc1(sw_trace, fs)

In [ ]:
# Plot spectrogram of the selected channel
plt.figure(figsize=(10, 4))
plt.pcolormesh(sw_times, sw_freqs, np.log10(sw_spec), shading="auto", cmap="viridis")
plt.yscale("log")
plt.colorbar(label="Power")

In [ ]:
# Plot PC1 (broadbandSlowWave score) over time
plt.figure(figsize=(10, 3), dpi=150)
plt.plot(sw_times, sw_pc1, lw=0.5)
plt.xlabel("Time (s)")
plt.ylabel("PC1 (slow-wave score)")
plt.title("abcGolden01 PC1 (ch{}, with CMR)".format(sw_channel))
sw_threshold = pc1_threshold(sw_pc1)
plt.axhline(sw_threshold, color="r", ls="--", label=f"threshold = {sw_threshold:.3f}")
plt.legend()

In [ ]:
# Step 5: threshold between the two modes via a 2-component Gaussian mixture.
# Fit the GMM here (same data + random_state as pc1_threshold, so it matches) to expose it for the plot below.
gmm = GaussianMixture(n_components=2, random_state=0).fit(sw_pc1.reshape(-1, 1))
sw_threshold = pc1_threshold(sw_pc1)
sw_nrem = sw_pc1 > sw_threshold  # NREM candidate per time bin

In [ ]:
# Plot PC1 distribution (histogram + KDE) with the fitted GMM components
plt.figure(figsize=(6, 4))
plt.hist(sw_pc1, bins=50, density=True, alpha=0.4)

x = np.linspace(sw_pc1.min(), sw_pc1.max(), 1000)
plt.plot(x, gaussian_kde(sw_pc1)(x), color="k", lw=1, label="KDE")
for i, (w, m, s) in enumerate(zip(gmm.weights_, gmm.means_.ravel(), np.sqrt(gmm.covariances_.ravel()))):
    plt.plot(x, w * np.exp(-0.5 * ((x - m) / s) ** 2) / (s * np.sqrt(2 * np.pi)), label=f"component {i}")

plt.axvline(sw_threshold, color="r", ls="--", label=f"threshold = {sw_threshold:.2f}")
plt.xlabel("PC1 (slow-wave score)")
plt.ylabel("density")
plt.legend()

### Check classification agreement between channels

Check correlation of binary classifications of sleep state between "best" channel (highest dip test score) and all other channels. Classifications are done by binarising > vs ≤ the threshold computed from a GMM fit with two Gaussians.

In [ ]:
# For each scanned channel: broadband PC1 -> GMM posterior-crossover threshold -> binary NREM series.
# Correlate each channel's binary series against the best channel's.

binary_by_channel = {}
for ci in candidate_idx:
    pc1 = broadband_pc1(rec.get_traces(channel_ids=[rec.channel_ids[ci]]).squeeze(), fs)[0]
    binary_by_channel[ci] = (pc1 > pc1_threshold(pc1)).astype(int)

ref_binary = binary_by_channel[best_channel]
corr_by_channel = {ci: np.corrcoef(b, ref_binary)[0, 1] for ci, b in binary_by_channel.items()}

In [ ]:
# Correlation vs channel, coloured by diptest quintile.
ch = np.array(sorted(corr_by_channel))
try:
    corr = np.array([corr_by_channel[c] for c in ch])
except:
    corr = np.load("/Users/willreith/Documents/phd/aeon/sleep_sandbox/notebooks/abcGolden01_binary-sleep-class_corr-with-best-channel_full-probe_channel-indexed.npy")
#dip_stats = np.load("/Users/willreith/Documents/phd/aeon/sleep_sandbox/notebooks/abcGolden01_hartigans-diptest_full-probe_channel-indexed.npy")
dips = dip_stats[ch]
quintile = np.digitize(dips, np.quantile(dips, [0.2, 0.4, 0.6, 0.8]))

plt.figure(figsize=(10, 3))
sc = plt.scatter(ch, corr, c=quintile, cmap="viridis")
plt.axvline(best_channel, color="r", ls="--", label=f"best = {best_channel}")
plt.colorbar(sc, label="diptest quintile", ticks=range(5))
plt.xlabel("channel index")
plt.ylabel("corr with best-channel binary")
plt.legend()

### Narrowband theta

In [ ]:
# Narrowband theta functions (THETA_BANDS, band_power, theta_ratio, select_theta_channel)
# are imported from sleep_sandbox.analysis -- see the imports cell.


In [ ]:
# Run both conventions on the CMR LFP (recording_lfp). Each picks its own best theta channel,
# then we take that channel's theta-ratio series. Time base matches sw_times (10 s / 1 s).
theta_series = {}
theta_channels = {}
theta_times = None
for conv in ("watson", "shin"):
    ch, dip, theta_candidates = select_theta_channel(recording_lfp, fs, conv, channel_stride)
    trace = recording_lfp.get_traces(channel_ids=[recording_lfp.channel_ids[ch]]).squeeze()
    spec, freqs, theta_times = log_spectrogram(trace, fs)
    theta_series[conv] = theta_ratio(spec, freqs, conv)
    theta_channels[conv] = ch
    print(f"{conv}: theta channel {ch} (dip={dip[ch]:.4f})")

In [ ]:
# Compare the two conventions: z-scored log-ratio over time, and against each other.
# z-score the log ratio (matches buzcode / the slow-wave metric; ratios are right-skewed).
z_watson = zscore(np.log10(theta_series["watson"]))
z_shin = zscore(np.log10(theta_series["shin"]))

fig, (ax_ts, ax_sc) = plt.subplots(1, 2, figsize=(13, 3), gridspec_kw={"width_ratios": [3, 1]})
ax_ts.plot(theta_times, z_watson, lw=0.5, label=f"watson (ch {theta_channels['watson']})")
ax_ts.plot(theta_times, z_shin, lw=0.5, label=f"shin (ch {theta_channels['shin']})")
ax_ts.set_xlabel("Time (s)")
ax_ts.set_ylabel("z-scored log theta ratio")
ax_ts.legend()

ax_sc.scatter(z_watson, z_shin, s=2, alpha=0.3)
r = np.corrcoef(z_watson, z_shin)[0, 1]
ax_sc.set_xlabel("watson")
ax_sc.set_ylabel("shin")
ax_sc.set_title(f"r = {r:.3f}")
fig.tight_layout()

In [ ]:
# Shin theta measure vs slow-wave PC1 (both z-scored for the overlay; same 10 s / 1 s time base).
z_shin = zscore(np.log10(theta_series["shin"]))
z_sw = zscore(sw_pc1)

# Correlation
plt.figure(figsize=(4, 4), dpi=150)
plt.scatter(z_sw, z_shin, s=2, alpha=0.3)
r = np.corrcoef(z_sw, z_shin)[0, 1]
plt.xlabel("slow-wave PC1 (z)")
plt.ylabel("shin theta (z)")
plt.title(f"r = {r:.3f}")

# Time series
plt.figure(figsize=(10, 3), dpi=150)
plt.plot(sw_times, z_sw, lw=0.5, label="slow-wave PC1")
plt.plot(theta_times, z_shin, lw=0.5, label=f"Shin theta (ch {theta_channels['shin']})")
plt.title("abcGolden01: slow-wave PC1 vs Shin theta (z-scored)")
plt.xlabel("Time (s)")
plt.ylabel("z-score")
plt.legend()

### IMU movement (BNO055) vs sleep metrics

Concatenate all BNO055 blocks, derive translational + angular speed and linear-acceleration
magnitude, and compare against slow-wave PC1 and the Shin narrowband-theta measure. Alignment (`align_bno055_to_lfp`): each block's IMU is placed on the LFP concat timeline via the
shared 250 MHz ONIX clock (`Bno055_Clock_N` ↔ `ProbeB_Clock_N`).

In [ ]:
# Align BNO055 to the LFP concat timeline via the shared 250 MHz ONIX clock
# (Bno055_Clock_N <-> ProbeB_Clock_N). Drops inter-block gaps + corrects the nominal-vs-true rate.
bno_dir = Path(os.getenv("RAW_DATA_DIR"))     # NeuropixelsV2 dir holding the Bno055 + ProbeB_Clock .bin files
imu_data, imu_t, imu_valid, imu_edges = align_bno055_to_lfp(bno_dir)
imu_speed, imu_ang, imu_accel = imu_kinematics(imu_data, imu_t, imu_edges)

# Duration check: aligned IMU timeline vs the LFP concat timeline (should now match closely).
print(f"IMU (aligned): {imu_t[imu_valid][-1]:.1f} s  ({imu_valid.sum()} valid / {imu_valid.size} samples)")
print(f"LFP sw_times:  {sw_times[-1]:.1f} s")
print(f"mismatch:      {sw_times[-1] - imu_t[imu_valid][-1]:.1f} s")

In [ ]:
# Reduce IMU (~100 Hz) to the 1 s PC1/theta grid: max per bin (catches brief movements).
# Only bin gap-valid samples (approach-B mask), and drop samples outside the LFP extent so the
# final segment's ~53 s clock overrun (IMU mapped beyond sw_times[-1]) doesn't pollute the edge bins.
tv = imu_t[imu_valid]
imu_speed_b = bin_max(tv, imu_speed[imu_valid], sw_times)
imu_ang_b = bin_max(tv, imu_ang[imu_valid], sw_times)
imu_accel_b = bin_max(tv, imu_accel[imu_valid], sw_times)

In [ ]:
# Figure 1: PC1, Shin theta, and IMU movement over the session (shared 1 s time base).
z_shin = zscore(np.log10(theta_series["shin"]))
lower = 6200
upper = 6500

fig, ax = plt.subplots(5, 1, sharex=True, figsize=(12, 9), dpi=150)
ax[0].plot(sw_times[lower:upper], sw_pc1[lower:upper], lw=0.5)
ax[0].axhline(sw_threshold, color="r", ls="--")
ax[0].set_ylabel("PC1 (slow wave)")
ax[1].plot(sw_times[lower:upper], z_shin[lower:upper], lw=0.5, color="C1")
ax[1].set_ylabel("shin theta (z)")
ax[2].plot(sw_times[lower:upper], imu_speed_b[lower:upper], lw=0.5, color="C2")
ax[2].set_ylabel("speed (m/s)")
ax[3].plot(sw_times[lower:upper], imu_ang_b[lower:upper], lw=0.5, color="C3")
ax[3].set_ylabel("ang. speed (deg/s)")
ax[4].plot(sw_times[lower:upper], imu_accel_b[lower:upper], lw=0.5, color="C4")
ax[4].set_ylabel("|accel| (m/s²)")
ax[4].set_xlabel("Time (s)")
fig.tight_layout()

In [ ]:
# Figure 2: each IMU movement signal vs each sleep metric (NaN bins dropped; r per panel).
movement = [("speed", imu_speed_b[lower:upper]), ("ang speed", imu_ang_b[lower:upper]), ("|accel|", imu_accel_b[lower:upper])]
metrics = [("PC1", sw_pc1[lower:upper]), ("shin theta (z)", z_shin[lower:upper])]

fig, axes = plt.subplots(len(movement), len(metrics), figsize=(9, 10))
for i, (mn, mv) in enumerate(movement):
    for j, (nn, nv) in enumerate(metrics):
        a = axes[i, j]
        ok = ~np.isnan(mv)
        a.scatter(nv[ok], mv[ok], s=2, alpha=0.3)
        r = np.corrcoef(nv[ok], mv[ok])[0, 1]
        a.set_title(f"r = {r:.2f}")
        if i == len(movement) - 1:
            a.set_xlabel(nn)
        if j == 0:
            a.set_ylabel(mn)
fig.tight_layout()

### Mismatch due to IMU data overrun

IMU data overruns amplifier data by ~53s at the end. This does not affect analysis, but here are some sanity checks that pins it outside of the main recording.

In [ ]:
# (1) Reconcile the true durations instead of comparing to sw_times[-1].
fs_raw = 30000  # nominal ephys sample rate (align_bno055_to_lfp default)

lfp_dur = recording_lfp.get_total_duration()
clock_blocks = sorted(int(p.stem.split("_")[-1]) for p in bno_dir.glob("*ProbeB_Clock_*.bin"))
clock_sum = sum(len(np.memmap(bno_dir / f"NeuropixelsV2_ProbeB_Clock_{b}.bin", dtype="<u8", mode="r")) / fs_raw
                for b in clock_blocks)
imu_end = imu_t[imu_valid][-1]

print(f"LFP recording total:       {lfp_dur:8.1f} s  ({recording_lfp.get_num_frames()} frames @ {recording_lfp.get_sampling_frequency():.0f} Hz)")
print(f"ProbeB_Clock sum (n={len(clock_blocks)}):   {clock_sum:8.1f} s")
print(f"aligned IMU end (valid):   {imu_end:8.1f} s")
print(f"sw_times[-1] (window ctr): {sw_times[-1]:8.1f} s")
print("---")
print(f"LFP - clock_sum:           {lfp_dur - clock_sum:+8.1f} s   (0 => derivative built from exactly these segments)")
print(f"LFP - aligned IMU end:     {lfp_dur - imu_end:+8.1f} s   (small +ve => IMU tail ends slightly early)")
print(f"LFP - sw_times[-1]:        {lfp_dur - sw_times[-1]:+8.1f} s   (~+5 s => STFT window-centre offset)")

In [ ]:
# (2) Event-lag cross-correlation: does movement line up with the neural wake proxy?
# Wake proxy = low slow-wave PC1 (-sw_pc1). Both on the 1 s grid; peak at lag 0 => truly aligned.
speed_z = zscore(np.nan_to_num(imu_speed_b[lower:upper], nan=0.0))
wake_z = zscore(-sw_pc1[lower:upper])
n = len(speed_z)
xc = np.correlate(speed_z, wake_z, mode="full") / n
lags = np.arange(-n + 1, n)                       # bins == seconds (1 s grid)
peak = int(lags[np.argmax(xc)])

win = 120
sel = np.abs(lags) <= win
plt.figure(figsize=(8, 3))
plt.plot(lags[sel], xc[sel], lw=0.8)
plt.axvline(peak, color="r", ls="--", label=f"peak lag = {peak} s")
plt.axvline(0, color="k", lw=0.5)
plt.xlabel("lag (s): speed relative to wake proxy")
plt.ylabel("cross-correlation")
plt.legend()
print(f"peak cross-correlation at lag {peak} s (r = {xc.max():.3f})")

### Intracranial EMG